In [1]:
#mo comment khi chua cai thu vien
# !pip install pandas sqlalchemy psycopg2-binary


## 1. Khai báo thư viện và cấu hình kết nối dùng chung


In [2]:
import pandas as pd
import psycopg2
import warnings
from sqlalchemy import create_engine

warnings.filterwarnings("ignore", category=UserWarning)

BRANCH_CONFIGS = [
    {
        "branch_name": "branch1",
        "branch_id": "BRANCH1",
        "host": "26.51.227.155",
        "port": 26257,
        "database": "btl2",
        "user": "root",
        "password": "",
        "schema": "public"
    },
    {
        "branch_name": "branch2",
        "branch_id": "BRANCH2",
        "host": "26.250.61.91",
        "port": 26257,
        "database": "btl2",
        "user": "root",
        "password": "",
        "schema": "public"
    },
    {
        "branch_name": "branch3",
        "branch_id": "BRANCH3",
        "host": "26.101.109.119",
        "port": 26257,
        "database": "btl2",
        "user": "root",
        "password": "",
        "schema": "public"
    }
]


### Query1. Tìm top 10 sản phẩm có doanh thu cao nhất trong tháng 05/2024 trên toàn hệ thống 3 chi nhánh

In [3]:
def read_branch_sales(branch_config, start_date, end_date):
    branch_name = branch_config["branch_name"]
    schema_name = branch_config["schema"]

    conn = None

    try:
        conn = psycopg2.connect(
            host=branch_config["host"],
            port=branch_config["port"],
            database=branch_config["database"],
            user=branch_config["user"],
            password=branch_config["password"],
            sslmode="disable",
            connect_timeout=5
        )

        query = f"""
            SELECT 
                %s AS source_branch,
                pi.product_id,
                pi.product_name,
                pi.category,
                pi.brand,
                oi.quantity,
                oi.total_price_usd,
                o.order_id
            FROM {schema_name}.orders o
            JOIN {schema_name}.order_items oi 
                ON o.order_id = oi.order_id
            JOIN {schema_name}.products_info pi 
                ON oi.product_id = pi.product_id
            WHERE 
                o.order_date >= DATE %s
                AND o.order_date < DATE %s
        """

        df = pd.read_sql(
            query,
            conn,
            params=(branch_name, start_date, end_date)
        )

        print(f"Đã đọc {len(df)} dòng từ {branch_name}")
        return df

    except Exception as e:
        print(f"Lỗi khi đọc dữ liệu từ {branch_name}: {e}")
        return pd.DataFrame()

    finally:
        if conn is not None:
            conn.close()

def distributed_top_products_query(start_date, end_date, top_n=10):
    branch_dfs = []

    for branch_config in BRANCH_CONFIGS:
        branch_name = branch_config["branch_name"]
        df = read_branch_sales(
            branch_config=branch_config,
            start_date=start_date,
            end_date=end_date
        )

        if not df.empty:
            branch_dfs.append(df)
        else:
            print(f"{branch_name} không có dữ liệu hoặc bị lỗi")

    if not branch_dfs:
        return pd.DataFrame()

    all_sales_df = pd.concat(
        branch_dfs,
        ignore_index=True
    )

    top_products_df = (
        all_sales_df
        .groupby(
            ["product_id", "product_name", "category", "brand"],
            as_index=False
        )
        .agg(
            total_quantity=("quantity", "sum"),
            total_revenue=("total_price_usd", "sum"),
            number_of_orders=("order_id", "nunique")
        )
        .sort_values("total_revenue", ascending=False)
        .head(top_n)
    )

    return top_products_df

top_products_df = distributed_top_products_query(
    "2024-05-01",
    "2024-06-01",
    10
)

display(top_products_df)

Đã đọc 14160 dòng từ branch1
Đã đọc 14368 dòng từ branch2
Đã đọc 14169 dòng từ branch3


,product_id,product_name,category,brand,total_quantity,total_revenue,number_of_orders
87,411,Phone Case Shockproof,Electronics,GoPro,877,207176.85,277
27,39,Wireless Bluetooth Earbuds,Electronics,Xiaomi,839,194132.50,282
42,111,Laptop Backpack Waterproof,Electronics,Xiaomi,767,192777.29,257
85,402,Smart Watch Series 8,Electronics,GoPro,759,190732.84,261
7,8,Phone Case Shockproof,Electronics,Samsung,771,188473.08,245
97,675,Wireless Mouse 2.4GHz,Electronics,HP,786,187526.35,257
113,693,Phone Case Shockproof,Electronics,JBL,815,186479.24,268
94,490,4K Webcam Pro,Electronics,Sony,756,185168.73,236
52,339,USB Hub 7-Port,Electronics,Apple,742,181840.41,244
99,677,Power Bank 20000mAh,Electronics,Dell,745,180685.87,245


## Query 2. Doanh thu theo phương thức thanh toán

**Ý nghĩa:** So sánh doanh thu và giá trị đơn hàng trung bình theo từng phương thức thanh toán ở các chi nhánh.


In [4]:
def read_branch_payment_revenue(branch_config):
    branch_name = branch_config["branch_name"]
    schema_name = branch_config["schema"]

    conn = None

    try:
        conn = psycopg2.connect(
            host=branch_config["host"],
            port=branch_config["port"],
            database=branch_config["database"],
            user=branch_config["user"],
            password=branch_config["password"],
            sslmode="disable",
            connect_timeout=5
        )

        query = f"""
            SELECT 
                %s AS source_branch,
                payment_method,
                total_price_usd
            FROM {schema_name}.orders
        """

        df = pd.read_sql(
            query,
            conn,
            params=(branch_name,)
        )

        print(f"Đã đọc {len(df)} dòng từ {branch_name}")
        return df

    except Exception as e:
        print(f"Lỗi khi đọc dữ liệu từ {branch_name}: {e}")
        return pd.DataFrame()

    finally:
        if conn is not None:
            conn.close()

In [5]:
def distributed_payment_revenue_query():
    branch_dfs = []

    for branch_config in BRANCH_CONFIGS:
        branch_name = branch_config["branch_name"]

        df = read_branch_payment_revenue(
            branch_config=branch_config
        )

        if not df.empty:
            branch_dfs.append(df)
        else:
            print(f"{branch_name} không có dữ liệu hoặc bị lỗi")

    if not branch_dfs:
        return pd.DataFrame()

    all_orders_df = pd.concat(
        branch_dfs,
        ignore_index=True
    )

    result_df = (
        all_orders_df
        .groupby(
            ["source_branch", "payment_method"],
            as_index=False
        )
        .agg(
            total_orders=("total_price_usd", "size"),
            total_revenue=("total_price_usd", "sum"),
            avg_order_value=("total_price_usd", "mean")
        )
    )

    result_df["avg_order_value"] = result_df["avg_order_value"].round(2)

    result_df = result_df.sort_values(
        ["source_branch", "total_revenue"],
        ascending=[True, False]
    )

    return result_df

In [6]:
display(distributed_payment_revenue_query())

Đã đọc 333374 dòng từ branch1
Đã đọc 333377 dòng từ branch2
Đã đọc 333375 dòng từ branch3


,source_branch,payment_method,total_orders,total_revenue,avg_order_value
3,branch1,Debit Card,67113,27009872.95,402.45
2,branch1,Credit Card,66427,26915458.86,405.19
0,branch1,Apple Pay,66877,26877003.61,401.89
1,branch1,Bank Transfer,66350,26805808.84,404.01
4,branch1,PayPal,66607,26751387.21,401.63
10,branch2,PayPal,66990,27189189.55,405.87
9,branch2,Debit Card,66956,27122409.11,405.08
6,branch2,Bank Transfer,66980,27049197.43,403.84
8,branch2,Credit Card,66268,26764470.12,403.88
5,branch2,Apple Pay,66179,26613585.37,402.15


## Query 3. Khách hàng mua ở cả 3 chi nhánh

**Ý nghĩa:** Tìm khách hàng có đơn hàng tại đủ tất cả chi nhánh.


In [7]:
def read_branch_order_customers(branch_config):
    branch_name = branch_config["branch_name"]
    schema_name = branch_config["schema"]

    conn = None

    try:
        conn = psycopg2.connect(
            host=branch_config["host"],
            port=branch_config["port"],
            database=branch_config["database"],
            user=branch_config["user"],
            password=branch_config["password"],
            sslmode="disable",
            connect_timeout=5
        )

        query = f"""
            SELECT DISTINCT customer_id
            FROM {schema_name}.orders
        """

        df = pd.read_sql(query, conn)

        print(f"Đã đọc {len(df)} customer_id từ {branch_name}")
        return df

    except Exception as e:
        print(f"Lỗi khi đọc customer_id từ {branch_name}: {e}")
        return pd.DataFrame()

    finally:
        if conn is not None:
            conn.close()

In [8]:
def read_customer_info_from_branch1(customer_ids):
    branch1_config = BRANCH_CONFIGS[0]
    schema_name = branch1_config["schema"]

    conn = None

    try:
        conn = psycopg2.connect(
            host=branch1_config["host"],
            port=branch1_config["port"],
            database=branch1_config["database"],
            user=branch1_config["user"],
            password=branch1_config["password"],
            sslmode="disable",
            connect_timeout=5
        )

        placeholders = ",".join(["%s"] * len(customer_ids))

        query = f"""
            SELECT 
                customer_id,
                customer_name,
                city,
                country
            FROM {schema_name}.customers
            WHERE customer_id IN ({placeholders})
            ORDER BY customer_id
        """

        df = pd.read_sql(
            query,
            conn,
            params=tuple(customer_ids)
        )

        print(f"Đã đọc thông tin {len(df)} khách hàng từ BRANCH1")
        return df

    except Exception as e:
        print(f"Lỗi khi đọc thông tin khách hàng từ BRANCH1: {e}")
        return pd.DataFrame()

    finally:
        if conn is not None:
            conn.close()

In [9]:
def distributed_customers_in_all_branches_query():
    customer_sets = []

    for branch_config in BRANCH_CONFIGS:
        branch_name = branch_config["branch_name"]

        df = read_branch_order_customers(
            branch_config=branch_config
        )

        if not df.empty:
            customer_ids = set(df["customer_id"].dropna().tolist())
            customer_sets.append(customer_ids)
        else:
            print(f"{branch_name} không có dữ liệu hoặc bị lỗi")

    if not customer_sets:
        return pd.DataFrame()

    common_customer_ids = set.intersection(*customer_sets)

    print(f"Số khách hàng mua ở tất cả chi nhánh: {len(common_customer_ids)}")

    if not common_customer_ids:
        return pd.DataFrame(
            columns=["customer_id", "customer_name", "city", "country"]
        )

    result_df = read_customer_info_from_branch1(common_customer_ids)

    return result_df

In [10]:
display(distributed_customers_in_all_branches_query())

Đã đọc 320609 customer_id từ branch1
Đã đọc 320609 customer_id từ branch2
Đã đọc 320523 customer_id từ branch3
Số khách hàng mua ở tất cả chi nhánh: 4855
Đã đọc thông tin 4855 khách hàng từ BRANCH1


,customer_id,customer_name,city,country
0,85,Michael Rodriguez,Ghent,Belgium
1,105,Matthew Smith,New York,United States
2,124,Mary Scott,New York,United States
3,127,Amy Williams,Bruges,Belgium
4,181,Ashley Johnson,Eindhoven,Netherlands
...,...,...,...,...
4850,333072,William Williams,Antwerp,Belgium
4851,333807,Melissa Davis,Montreal,Canada
4852,333877,Ashley Martin,Berlin,Germany
4853,338690,Ashley Perez,Munich,Germany


## Query 4. Sản phẩm có tồn kho ở chi nhánh 1 nhưng chưa có ở chi nhánh 2

**Ý nghĩa:** Tìm sản phẩm đang có hàng tại BRANCH1 nhưng BRANCH2 không có hàng.



In [11]:
def read_branch_stock_products(branch_config):
    branch_name = branch_config["branch_name"]
    schema_name = branch_config["schema"]

    conn = None

    try:
        conn = psycopg2.connect(
            host=branch_config["host"],
            port=branch_config["port"],
            database=branch_config["database"],
            user=branch_config["user"],
            password=branch_config["password"],
            sslmode="disable",
            connect_timeout=5
        )

        query = f"""
            SELECT DISTINCT product_id
            FROM {schema_name}.products_stock
            WHERE stock_quantity > 0
        """

        df = pd.read_sql(query, conn)

        print(f"Đã đọc {len(df)} sản phẩm tồn kho từ {branch_name}")
        return df

    except Exception as e:
        print(f"Lỗi khi đọc tồn kho từ {branch_name}: {e}")
        return pd.DataFrame()

    finally:
        if conn is not None:
            conn.close()

In [12]:
def read_product_info_from_branch1(product_ids):
    branch1_config = BRANCH_CONFIGS[0]
    schema_name = branch1_config["schema"]

    conn = None

    try:
        conn = psycopg2.connect(
            host=branch1_config["host"],
            port=branch1_config["port"],
            database=branch1_config["database"],
            user=branch1_config["user"],
            password=branch1_config["password"],
            sslmode="disable",
            connect_timeout=5
        )

        placeholders = ",".join(["%s"] * len(product_ids))

        query = f"""
            SELECT 
                product_id,
                product_name,
                category,
                brand
            FROM {schema_name}.products_info
            WHERE product_id IN ({placeholders})
            ORDER BY product_id
        """

        df = pd.read_sql(
            query,
            conn,
            params=tuple(product_ids)
        )

        print(f"Đã đọc thông tin {len(df)} sản phẩm từ BRANCH1")
        return df

    except Exception as e:
        print(f"Lỗi khi đọc thông tin sản phẩm từ BRANCH1: {e}")
        return pd.DataFrame()

    finally:
        if conn is not None:
            conn.close()

In [13]:
def distributed_stock_branch1_not_branch2_query():
    branch1_config = BRANCH_CONFIGS[0]
    branch2_config = BRANCH_CONFIGS[1]

    branch1_stock_df = read_branch_stock_products(branch1_config)
    branch2_stock_df = read_branch_stock_products(branch2_config)

    if branch1_stock_df.empty:
        print("BRANCH1 không có dữ liệu tồn kho")
        return pd.DataFrame()

    branch1_products = set(branch1_stock_df["product_id"].dropna().tolist())

    if branch2_stock_df.empty:
        branch2_products = set()
    else:
        branch2_products = set(branch2_stock_df["product_id"].dropna().tolist())

    diff_product_ids = branch1_products - branch2_products

    print(f"Số sản phẩm có ở BRANCH1 nhưng chưa có ở BRANCH2: {len(diff_product_ids)}")

    if not diff_product_ids:
        return pd.DataFrame(
            columns=["product_id", "product_name", "category", "brand"]
        )

    result_df = read_product_info_from_branch1(diff_product_ids)

    return result_df

In [14]:
display(distributed_stock_branch1_not_branch2_query())

Đã đọc 998 sản phẩm tồn kho từ branch1
Đã đọc 1004 sản phẩm tồn kho từ branch2
Số sản phẩm có ở BRANCH1 nhưng chưa có ở BRANCH2: 0


,product_id,product_name,category,brand


## Query 5. Sản phẩm đã bán ở tất cả 3 chi nhánh

**Ý nghĩa:** Tìm sản phẩm phổ biến toàn hệ thống, tức là có phát sinh bán hàng ở đủ các chi nhánh.



In [15]:
def read_branch_sold_products(branch_config):
    branch_name = branch_config["branch_name"]
    schema_name = branch_config["schema"]

    conn = None

    try:
        conn = psycopg2.connect(
            host=branch_config["host"],
            port=branch_config["port"],
            database=branch_config["database"],
            user=branch_config["user"],
            password=branch_config["password"],
            sslmode="disable",
            connect_timeout=5
        )

        query = f"""
            SELECT DISTINCT oi.product_id
            FROM {schema_name}.order_items oi
            JOIN {schema_name}.orders o
                ON oi.order_id = o.order_id
        """

        df = pd.read_sql(query, conn)

        print(f"Đã đọc {len(df)} sản phẩm đã bán từ {branch_name}")
        return df

    except Exception as e:
        print(f"Lỗi khi đọc sản phẩm đã bán từ {branch_name}: {e}")
        return pd.DataFrame()

    finally:
        if conn is not None:
            conn.close()

In [16]:
def distributed_products_sold_in_all_branches_query():
    product_sets = []

    for branch_config in BRANCH_CONFIGS:
        branch_name = branch_config["branch_name"]

        df = read_branch_sold_products(
            branch_config=branch_config
        )

        if not df.empty:
            product_ids = set(df["product_id"].dropna().tolist())
            product_sets.append(product_ids)
        else:
            print(f"{branch_name} không có dữ liệu hoặc bị lỗi")

    if not product_sets:
        return pd.DataFrame()

    common_product_ids = set.intersection(*product_sets)

    print(f"Số sản phẩm đã bán ở tất cả chi nhánh: {len(common_product_ids)}")

    if not common_product_ids:
        return pd.DataFrame(
            columns=["product_id", "product_name", "category", "brand"]
        )

    result_df = read_product_info_from_branch1(common_product_ids)

    return result_df

In [17]:
display(distributed_products_sold_in_all_branches_query())

Đã đọc 49 sản phẩm đã bán từ branch1
Đã đọc 52 sản phẩm đã bán từ branch2
Đã đọc 49 sản phẩm đã bán từ branch3
Số sản phẩm đã bán ở tất cả chi nhánh: 1
Đã đọc thông tin 1 sản phẩm từ BRANCH1


,product_id,product_name,category,brand
0,10,Massage Gun Deep Tissue,Health,Nikon


### Query 6

Hàm đọc dữ liệu cục bộ từ các mảnh chi nhánh

In [18]:
def read_local_branch_revenue(config):
    """
    Kết nối trực tiếp tới IP của một Node, truy vấn lấy dữ liệu thô
    đã kết hợp giữa Orders và Employees của chi nhánh đó.
    """
    bid = config["branch_id"]
    schema = config["schema"]

    # Câu lệnh SQL cục bộ: Sử dụng chữ thường hoàn toàn, không bọc nháy kép đối tượng
    query = f"""
        SELECT 
            e.employee_id,
            e.first_name || ' ' || e.last_name AS employee_name,
            o.order_id,
            o.total_price_usd
        FROM {schema}.orders o
        JOIN {schema}.employees e ON o.employee_id = e.employee_id
    """

    conn = None
    try:
        # Thiết lập kết nối trực tiếp tới IP máy chi nhánh nguồn
        conn = psycopg2.connect(
            host=config["host"],
            port=config["port"],
            database=config["database"],
            user=config["user"],
            password=config["password"],
            sslmode="disable",
            connect_timeout=5
        )

        # Đọc dữ liệu trực tiếp qua kết nối psycopg2
        df = pd.read_sql_query(query, conn)

        # Thầy cô chấm phân tán sẽ cần cột định danh nguồn gốc dữ liệu này
        df.insert(0, "source_branch", bid)
        print(f"-> Thành công: Đã tải {len(df)} dòng dữ liệu từ {bid} ({config['host']})")
        return df

    except Exception as e:
        print(f"-> Thất bại: Không thể lấy dữ liệu từ {bid} tại IP {config['host']}. Chi tiết lỗi: {e}")
        return pd.DataFrame()

    finally:
        if conn is not None:
            conn.close()

Bộ xử lý tích hợp dữ liệu toàn cục - Global Query Engine

In [19]:
def execute_distributed_query_6(revenue_threshold=50000):
    local_dfs = []

    # Duyệt qua từng cấu hình máy chủ chi nhánh để kéo dữ liệu thô về bộ nhớ RAM local
    for config in BRANCH_CONFIGS:
        df_branch = read_local_branch_revenue(config)
        if not df_branch.empty:
            local_dfs.append(df_branch)

    if not local_dfs:
        print("LỖI TOÀN CỤC: Không lấy được dữ liệu từ bất kỳ máy chi nhánh nào!")
        return pd.DataFrame()

    # Mô phỏng phép toán toán tử UNION ALL hệ thống phân tán
    global_raw_data = pd.concat(local_dfs, ignore_index=True)

    # Xử lý tổng hợp dữ liệu (GROUP BY) trên bộ nhớ máy chủ điều phối trung tâm
    global_result = (
        global_raw_data
        .groupby(["source_branch", "employee_id", "employee_name"], as_index=False)
        .agg(
            total_orders=("order_id", "count"),
            total_revenue=("total_price_usd", "sum"),
            avg_order_value=("total_price_usd", "mean")
        )
    )

    # Làm tròn giá trị trung bình đơn hàng theo định dạng tiền tệ
    global_result["avg_order_value"] = global_result["avg_order_value"].round(2)

    # Lọc điều kiện tích lũy doanh số lớn hơn hoặc bằng ngưỡng thiết lập (Tương đương HAVING)
    final_kpi_df = global_result[global_result["total_revenue"] >= revenue_threshold]

    # Sắp xếp thứ hạng doanh thu từ cao xuống thấp (Tương đương ORDER BY ... DESC)
    final_kpi_df = final_kpi_df.sort_values(by="total_revenue", ascending=False)

    return final_kpi_df

Thực thi câu hỏi truy vấn số 6 và xuất báo cáo

In [20]:
# Thực thi hàm xử lý gom dữ liệu với ngưỡng KPI 50,000 USD
report_df = execute_distributed_query_6(revenue_threshold=50000)

# Kiểm tra trạng thái dữ liệu và xuất báo cáo trực quan dạng bảng
if not report_df.empty:
    print("\n--- KẾT QUẢ ĐÁNH GIÁ KPI NHÂN VIÊN TRÊN TOÀN HỆ THỐNG PHÂN TÁN ---")
    display(report_df)
else:
    print("Không có nhân viên nào đạt đủ chỉ tiêu doanh số lọc.")

-> Thành công: Đã tải 333374 dòng dữ liệu từ BRANCH1 (26.51.227.155)
-> Thành công: Đã tải 333377 dòng dữ liệu từ BRANCH2 (26.250.61.91)
-> Thành công: Đã tải 333375 dòng dữ liệu từ BRANCH3 (26.101.109.119)

--- KẾT QUẢ ĐÁNH GIÁ KPI NHÂN VIÊN TRÊN TOÀN HỆ THỐNG PHÂN TÁN ---


,source_branch,employee_id,employee_name,total_orders,total_revenue,avg_order_value
60,BRANCH1,61,Tu Phan,3434,1402512.56,408.42
96,BRANCH1,97,Frank Jones,3336,1391541.60,417.13
72,BRANCH1,73,Dung Do,3401,1390991.41,408.99
21,BRANCH1,22,Phuc Bui,3378,1388833.41,411.14
68,BRANCH1,69,Binh Jones,3441,1388128.64,403.41
...,...,...,...,...,...,...
188,BRANCH2,439,Charlotte Smith,932,346740.30,372.04
543,BRANCH3,792,Paul Garcia,888,344775.09,388.26
400,BRANCH2,651,Michael Brown,895,342995.11,383.23
712,BRANCH3,961,Steven Harris,899,340754.00,379.04


### Query 7

Hàm đọc dữ liệu giảm giá sản phẩm cục bộ tại chi nhánh

In [21]:
def read_local_branch_discount(config):
    """
    Kết nối trực tiếp tới IP của một Node, truy vấn lấy dữ liệu thô
    của Order Items kết hợp thông tin danh mục sản phẩm (Products Info).
    """
    bid = config["branch_id"]
    schema = config["schema"]

    # Câu lệnh SQL cục bộ: Lấy các trường phục vụ tính toán mức giảm giá
    query = f"""
        SELECT 
            pi.category, 
            oi.quantity, 
            oi.discount_percent,
            oi.discount_amount_usd, 
            oi.total_price_usd
        FROM {schema}.order_items oi
        JOIN {schema}.products_info pi ON oi.product_id = pi.product_id
    """

    conn = None
    try:
        conn = psycopg2.connect(
            host=config["host"],
            port=config["port"],
            database=config["database"],
            user=config["user"],
            password=config["password"],
            sslmode="disable",
            connect_timeout=5
        )

        df = pd.read_sql_query(query, conn)
        df.insert(0, "source_branch", bid)
        print(f"-> Câu 7 - Thành công: Đã tải {len(df)} dòng dữ liệu từ {bid} ({config['host']})")
        return df

    except Exception as e:
        print(f"-> Câu 7 - Thất bại: Không thể lấy dữ liệu từ {bid} tại IP {config['host']}. Chi tiết lỗi: {e}")
        return pd.DataFrame()

    finally:
        if conn is not None:
            conn.close()

Bộ xử lý tích hợp toàn cục cho Câu 7

In [22]:
def execute_distributed_query_7(discount_threshold=8):
    local_dfs = []

    # Gom dữ liệu thô từ mạng lưới các chi nhánh
    for config in BRANCH_CONFIGS:
        df_branch = read_local_branch_discount(config)
        if not df_branch.empty:
            local_dfs.append(df_branch)

    if not local_dfs:
        print("LỖI TOÀN CỤC: Không lấy được dữ liệu từ bất kỳ máy chi nhánh nào!")
        return pd.DataFrame()

    # Thực hiện phép toán UNION ALL
    global_raw_data = pd.concat(local_dfs, ignore_index=True)

    # ==========================================
    # BƯỚC THÊM MỚI: ÉP KIỂU DỮ LIỆU SANG SỐ
    # ==========================================
    numeric_columns = [
        "quantity", 
        "discount_percent", 
        "discount_amount_usd", 
        "total_price_usd"
    ]
    
    for col in numeric_columns:
        # errors='coerce' sẽ tự động chuyển các chuỗi không phải số (nếu có) thành NaN để không gây lỗi
        global_raw_data[col] = pd.to_numeric(global_raw_data[col], errors='coerce')
    # ==========================================

    # Thực hiện tính toán gộp (GROUP BY category) toàn cục
    global_result = (
        global_raw_data
        .groupby("category", as_index=False)
        .agg(
            total_order_lines=("category", "count"),
            total_quantity_sold=("quantity", "sum"),
            avg_discount_percent=("discount_percent", "mean"),
            total_discount_amount=("discount_amount_usd", "sum"),
            total_revenue=("total_price_usd", "sum")
        )
    )

    # Làm tròn các cột số liệu theo định dạng chuẩn 2 chữ số thập phân
    global_result["avg_discount_percent"] = global_result["avg_discount_percent"].round(2)
    global_result["total_discount_amount"] = global_result["total_discount_amount"].round(2)
    global_result["total_revenue"] = global_result["total_revenue"].round(2)

    # Áp dụng mệnh đề HAVING: Lọc các loại sản phẩm có mức giảm giá trung bình >= 8%
    final_discount_df = global_result[global_result["avg_discount_percent"] >= discount_threshold]

    # Sắp xếp thứ hạng giảm giá từ cao xuống thấp (ORDER BY avg_discount_percent DESC)
    final_discount_df = final_discount_df.sort_values(by="avg_discount_percent", ascending=False)

    return final_discount_df

Thực thi câu hỏi truy vấn số 7 và hiển thị kết quả

In [23]:
# Chạy hàm xử lý phân tán câu 7 với ngưỡng lọc giảm giá là 8%
discount_report_df = execute_distributed_query_7(discount_threshold=8)

# Kiểm tra trạng thái dữ liệu và xuất kết quả
if not discount_report_df.empty:
    print("\n--- KẾT QUẢ PHÂN TÍCH NHÓM SẢN PHẨM GIẢM GIÁ CAO TOÀN HỆ THỐNG ---")
    display(discount_report_df)
else:
    print("Không có danh mục sản phẩm nào có mức giảm giá trung bình vượt ngưỡng chỉ định.")

-> Câu 7 - Thành công: Đã tải 333377 dòng dữ liệu từ BRANCH1 (26.51.227.155)
-> Câu 7 - Thành công: Đã tải 333378 dòng dữ liệu từ BRANCH2 (26.250.61.91)
-> Câu 7 - Thành công: Đã tải 333376 dòng dữ liệu từ BRANCH3 (26.101.109.119)

--- KẾT QUẢ PHÂN TÍCH NHÓM SẢN PHẨM GIẢM GIÁ CAO TOÀN HỆ THỐNG ---


,category,total_order_lines,total_quantity_sold,avg_discount_percent,total_discount_amount,total_revenue
1,Clothing,199544,598163,8.51,4189734.35,4.520584e+07
2,Electronics,200019,599007,8.51,13029031.16,1.398660e+08
4,Home,200592,601264,8.51,8181585.40,8.784173e+07
3,Health,199576,597996,8.49,5325138.43,5.746797e+07
7,Sports,200397,601321,8.49,6756320.26,7.291752e+07


### Query 8

Hàm đọc dữ liệu đơn hàng cục bộ từ các mảnh chi nhánh

In [24]:
def read_local_branch_orders(config):
    """
    Kết nối trực tiếp tới IP của một Node, truy vấn lấy dữ liệu thô
    của bảng Orders từ chi nhánh đó.
    """
    bid = config["branch_id"]
    schema = config["schema"]

    # Câu lệnh SQL cục bộ: Lấy thông tin các đơn hàng
    query = f"""
        SELECT 
            order_id, 
            order_date, 
            customer_id, 
            employee_id, 
            payment_method, 
            total_price_usd
        FROM {schema}.orders
    """

    conn = None
    try:
        conn = psycopg2.connect(
            host=config["host"],
            port=config["port"],
            database=config["database"],
            user=config["user"],
            password=config["password"],
            sslmode="disable",
            connect_timeout=5
        )

        df = pd.read_sql_query(query, conn)
        df.insert(0, "source_branch", bid)
        print(f"-> Câu 8 - Thành công: Đã tải {len(df)} đơn hàng từ {bid} ({config['host']})")
        return df

    except Exception as e:
        print(f"-> Câu 8 - Thất bại: Không thể lấy dữ liệu từ {bid} tại IP {config['host']}. Chi tiết lỗi: {e}")
        return pd.DataFrame()

    finally:
        if conn is not None:
            conn.close()

Bộ xử lý phân tích đơn hàng bất thường toàn cục - Global Query Engine

In [25]:
def execute_distributed_query_8():
    local_dfs = []

    # Gom toàn bộ đơn hàng thô từ mạng lưới phân tán
    for config in BRANCH_CONFIGS:
        df_branch = read_local_branch_orders(config)
        if not df_branch.empty:
            local_dfs.append(df_branch)

    if not local_dfs:
        print("LỖI TOÀN CỤC: Không lấy được dữ liệu từ bất kỳ máy chi nhánh nào!")
        return pd.DataFrame()

    # Tương đương phép UNION ALL (Biến all_orders trong câu lệnh SQL)
    all_orders = pd.concat(local_dfs, ignore_index=True)

    # Tương đương bảng tạm branch_avg trong SQL: Tính giá trị trung bình đơn hàng của TỪNG chi nhánh
    branch_avg = (
        all_orders
        .groupby("source_branch")["total_price_usd"]
        .mean()
        .reset_index(name="avg_branch_order_value")
    )

    # Tương đương phép JOIN: Tích hợp giá trị trung bình chi nhánh vào từng đơn hàng chi tiết
    merged_df = pd.merge(all_orders, branch_avg, on="source_branch", how="inner")

    # Tương đương mệnh đề WHERE: Lọc các đơn hàng cao hơn 2 lần trung bình chi nhánh đó
    anomaly_df = merged_df[merged_df["total_price_usd"] > (merged_df["avg_branch_order_value"] * 2)].copy()

    if anomaly_df.empty:
        return pd.DataFrame()

    # Tính toán tỷ lệ vượt ngưỡng (ratio_to_avg) và làm tròn số liệu 2 chữ số thập phân
    anomaly_df["ratio_to_avg"] = (anomaly_df["total_price_usd"] / anomaly_df["avg_branch_order_value"]).round(2)
    anomaly_df["avg_branch_order_value"] = anomaly_df["avg_branch_order_value"].round(2)
    anomaly_df["total_price_usd"] = anomaly_df["total_price_usd"].round(2)

    # Tương đương mệnh đề ORDER BY ratio_to_avg DESC
    final_report_df = anomaly_df.sort_values(by="ratio_to_avg", ascending=False)

    return final_report_df

Thực thi câu hỏi truy vấn số 8 và xuất kết quả

In [26]:
# Chạy hàm phân tích phát hiện đơn hàng bất thường toàn hệ thống
anomaly_report_df = execute_distributed_query_8()

# Kiểm tra và xuất bảng báo cáo trực quan
if not anomaly_report_df.empty:
    print("\n--- CẢNH BÁO: DANH SÁCH ĐƠN HÀNG LỚN BẤT THƯỜNG (> 2 LẦN TRUNG BÌNH CHI NHÁNH) ---")
    display(anomaly_report_df)
else:
    print("Hệ thống an toàn: Không phát hiện đơn hàng nào có giá trị lớn bất thường.")

-> Câu 8 - Thành công: Đã tải 333374 đơn hàng từ BRANCH1 (26.51.227.155)
-> Câu 8 - Thành công: Đã tải 333377 đơn hàng từ BRANCH2 (26.250.61.91)
-> Câu 8 - Thành công: Đã tải 333375 đơn hàng từ BRANCH3 (26.101.109.119)

--- CẢNH BÁO: DANH SÁCH ĐƠN HÀNG LỚN BẤT THƯỜNG (> 2 LẦN TRUNG BÌNH CHI NHÁNH) ---


,source_branch,order_id,order_date,customer_id,employee_id,payment_method,total_price_usd,avg_branch_order_value,ratio_to_avg
1000,BRANCH1,1001,2024-02-05 09:37:55+00:00,213056,75,Debit Card,11095.68,403.03,27.53
848795,BRANCH3,875733,2025-05-06,746935,908,Apple Pay,2498.80,402.55,6.21
991853,BRANCH3,906825,2025-07-13,994267,794,Bank Transfer,2498.75,402.55,6.21
893351,BRANCH3,963560,2025-11-14,746289,1046,Bank Transfer,2495.60,402.55,6.20
747006,BRANCH3,742826,2024-07-18,897474,984,PayPal,2496.05,402.55,6.20
...,...,...,...,...,...,...,...,...,...
543180,BRANCH2,543181,2025-05-08 16:38:27+00:00,107655,694,Bank Transfer,809.75,404.17,2.00
164537,BRANCH1,164538,2025-01-28 08:02:42+00:00,176487,89,Credit Card,806.90,403.03,2.00
401840,BRANCH2,401841,2024-07-01 13:52:37+00:00,414875,501,PayPal,808.72,404.17,2.00
651446,BRANCH2,651447,2025-12-31 04:56:37+00:00,645024,588,Credit Card,808.99,404.17,2.00


### Query 9

Hàm đọc dữ liệu bán hàng và tồn kho cục bộ tại chi nhánh

In [27]:
def read_local_branch_sales_and_stock(config):
    """
    Kết nối tới IP chi nhánh để lấy dữ liệu bán hàng đã gộp nhóm 
    và dữ liệu tồn kho của các sản phẩm tại chi nhánh đó.
    """
    bid = config["branch_id"]
    schema = config["schema"]

    # SQL 1: Tính tổng số lượng đã bán theo từng product_id tại chi nhánh
    sales_query = f"""
        SELECT product_id, SUM(quantity::NUMERIC) AS sold_quantity
        FROM {schema}.order_items
        GROUP BY product_id
    """
    
    # SQL 2: Tính tổng số lượng tồn kho theo từng product_id tại chi nhánh
    stock_query = f"""
        SELECT product_id, SUM(stock_quantity::NUMERIC) AS stock_quantity
        FROM {schema}.products_stock
        GROUP BY product_id
    """

    conn = None
    try:
        conn = psycopg2.connect(
            host=config["host"],
            port=config["port"],
            database=config["database"],
            user=config["user"],
            password=config["password"],
            sslmode="disable",
            connect_timeout=5
        )

        # Đọc dữ liệu thô cục bộ sang Dataframe
        df_sales = pd.read_sql_query(sales_query, conn)
        df_stock = pd.read_sql_query(stock_query, conn)
        
        print(f"-> Câu 9 - Thành công: Tải dữ liệu phân mảnh từ {bid} ({config['host']})")
        return df_sales, df_stock

    except Exception as e:
        print(f"-> Câu 9 - Thất bại: Lỗi tại mảnh {bid} ({config['host']}). Chi tiết: {e}")
        return pd.DataFrame(), pd.DataFrame()

    finally:
        if conn is not None:
            conn.close()

Hàm đọc bảng thông tin sản phẩm nền từ chi nhánh điều phối

In [28]:
def read_global_products_info():
    """
    Kéo bảng thông tin chi tiết sản phẩm (bảng danh mục nền) từ máy BRANCH1.
    """
    # Tìm cấu hình của BRANCH1 trong danh sách hệ thống
    config_b1 = next((c for c in BRANCH_CONFIGS if c["branch_id"] == "BRANCH1"), BRANCH_CONFIGS[0])
    schema = config_b1["schema"]
    
    query = f"""
        SELECT product_id, product_name, category, brand 
        FROM {schema}.products_info
    """
    conn = None
    try:
        conn = psycopg2.connect(
            host=config_b1["host"],
            port=config_b1["port"],
            database=config_b1["database"],
            user=config_b1["user"],
            password=config_b1["password"],
            sslmode="disable",
            connect_timeout=5
        )
        return pd.read_sql_query(query, conn)
    except Exception as e:
        print(f"❌ Không thể lấy bảng danh mục PRODUCTS_INFO nền từ BRANCH1: {e}")
        return pd.DataFrame()
    finally:
        if conn is not None:
            conn.close()

Bộ xử lý phân tích cảnh báo tồn kho toàn cục - Global Query Engine

In [29]:
def execute_distributed_query_9():
    all_sales_list = []
    all_stock_list = []

    # 1. Thu thập các mảnh dữ liệu bán hàng và tồn kho từ mạng lưới phân tán
    for config in BRANCH_CONFIGS:
        df_sales, df_stock = read_local_branch_sales_and_stock(config)
        if not df_sales.empty:
            all_sales_list.append(df_sales)
        if not df_stock.empty:
            all_stock_list.append(df_stock)

    if not all_sales_list:
        print("LỖI TOÀN CỤC: Không kéo được dữ liệu bán hàng từ bất kỳ chi nhánh nào!")
        return pd.DataFrame()

    # 2. Thực hiện phép toán UNION ALL tương đương SQL cho cả 2 nhóm dữ liệu
    raw_sales_union = pd.concat(all_sales_list, ignore_index=True)
    raw_stock_union = pd.concat(all_stock_list, ignore_index=True) if all_stock_list else pd.DataFrame(columns=["product_id", "stock_quantity"])

    # 3. Tính tổng lượng bán toàn hệ thống (total_sales)
    total_sales = (
        raw_sales_union
        .groupby("product_id", as_index=False)["sold_quantity"]
        .sum()
        .rename(columns={"sold_quantity": "total_sold_quantity"})
    )

    # 4. Tính tổng lượng tồn kho toàn hệ thống (total_stock)
    total_stock = (
        raw_stock_union
        .groupby("product_id", as_index=False)["stock_quantity"]
        .sum()
        .rename(columns={"stock_quantity": "total_stock_quantity"})
    )

    # 5. LEFT JOIN dữ liệu bán và tồn kho, đồng thời xử lý giá trị rỗng (tương đương NVL)
    merged_kpi = pd.merge(total_sales, total_stock, on="product_id", how="left")
    merged_kpi["total_stock_quantity"] = merged_kpi["total_stock_quantity"].fillna(0).astype(int)

    # 6. Định nghĩa logic CASE WHEN trong SQL để gán nhãn trạng thái kho (stock_status)
    def determine_stock_status(row):
        stock = row["total_stock_quantity"]
        sold = row["total_sold_quantity"]
        if stock == 0:
            return "OUT_OF_STOCK"
        elif stock < sold * 0.2:
            return "CRITICAL"
        elif stock < sold * 0.5:
            return "LOW"
        else:
            return "NORMAL"

    merged_kpi["stock_status"] = merged_kpi.apply(determine_stock_status, axis=1)

    # 7. Áp dụng mệnh đề WHERE: Chỉ lấy những sản phẩm có lượng tồn kho dưới 50% lượng đã bán (LOW, CRITICAL, OUT_OF_STOCK)
    filtered_kpi = merged_kpi[merged_kpi["total_stock_quantity"] < (merged_kpi["total_sold_quantity"] * 0.5)].copy()

    if filtered_kpi.empty:
        print("Hệ thống an toàn: Không có sản phẩm nào bị thiếu hụt tồn kho.")
        return pd.DataFrame()

    # 8. Kéo bảng thông tin sản phẩm nền về để JOIN lấy tên chi tiết (product_name, category, brand)
    df_products_info = read_global_products_info()
    if df_products_info.empty:
        return filtered_kpi # Trả về bảng mã nếu không lấy được bảng nền tên sản phẩm

    final_report = pd.merge(df_products_info, filtered_kpi, on="product_id", how="inner")

    # 9. Tương đương mệnh đề ORDER BY ts.total_sold_quantity DESC
    final_report = final_report.sort_values(by="total_sold_quantity", ascending=False)

    return final_report

Khởi chạy câu hỏi truy vấn số 9 và xuất báo cáo

In [30]:
# Thực thi câu truy vấn phân tích cảnh báo nhập hàng toàn cầu
stock_warning_df = execute_distributed_query_9()

# Hiển thị bảng kết quả trực quan
if not stock_warning_df.empty:
    print("\n--- BÁO CÁO CẢNH BÁO: CÁC SẢN PHẨM BÁN CHẠY CẦN ƯU TIÊN NHẬP THÊM HÀNG ---")
    display(stock_warning_df)

-> Câu 9 - Thành công: Tải dữ liệu phân mảnh từ BRANCH1 (26.51.227.155)
-> Câu 9 - Thành công: Tải dữ liệu phân mảnh từ BRANCH2 (26.250.61.91)
-> Câu 9 - Thành công: Tải dữ liệu phân mảnh từ BRANCH3 (26.101.109.119)

--- BÁO CÁO CẢNH BÁO: CÁC SẢN PHẨM BÁN CHẠY CẦN ƯU TIÊN NHẬP THÊM HÀNG ---


,product_id,product_name,category,brand,total_sold_quantity,total_stock_quantity,stock_status
81,387,Water Bottle Insulated,Sports,Sony,23432.0,756,CRITICAL
143,815,Tennis Balls Pack,Sports,JBL,23000.0,2226,CRITICAL
44,118,Cotton T-Shirt Pack,Clothing,HP,22853.0,1545,CRITICAL
141,796,Desk Lamp LED,Home,Logitech,22749.0,2484,CRITICAL
24,33,Vacuum Robot Cleaner,Home,Philips,22708.0,483,CRITICAL
...,...,...,...,...,...,...,...
18,24,4K Webcam Pro,Electronics,Dell,16338.0,1338,CRITICAL
29,49,Portable SSD 1TB,Electronics,Xiaomi,16286.0,1569,CRITICAL
4,5,Screen Protector Tempered Glass,Electronics,Anker,16238.0,246,CRITICAL
111,691,Smart Watch Series 8,Electronics,Adidas,16191.0,1359,CRITICAL


### Query 10

Hàm đọc và tính doanh thu sản phẩm cục bộ tại chi nhánh

In [31]:
def read_local_branch_product_revenue(config):
    """
    Kết nối trực tiếp tới IP của một Node, tính tổng doanh thu và số lượng
    đã bán của từng sản phẩm tại riêng chi nhánh đó.
    """
    bid = config["branch_id"]
    schema = config["schema"]

    # Câu lệnh SQL cục bộ: Gom nhóm tính tổng số lượng và doanh thu theo product_id
    query = f"""
        SELECT 
            oi.product_id,
            SUM(oi.quantity::NUMERIC) AS total_quantity,
            SUM(oi.total_price_usd::NUMERIC) AS total_revenue
        FROM {schema}.order_items oi
        JOIN {schema}.orders o ON oi.order_id = o.order_id
        GROUP BY oi.product_id
    """

    conn = None
    try:
        conn = psycopg2.connect(
            host=config["host"],
            port=config["port"],
            database=config["database"],
            user=config["user"],
            password=config["password"],
            sslmode="disable",
            connect_timeout=5
        )

        df = pd.read_sql_query(query, conn)
        df.insert(0, "source_branch", bid)
        print(f"-> Câu 10 - Thành công: Đã tải và tính toán xong doanh số sản phẩm từ {bid} ({config['host']})")
        return df

    except Exception as e:
        print(f"-> Câu 10 - Thất bại: Không thể lấy dữ liệu từ {bid} tại IP {config['host']}. Chi tiết lỗi: {e}")
        return pd.DataFrame()

    finally:
        if conn is not None:
            conn.close()

Hàm kéo bảng thông tin sản phẩm nền từ chi nhánh trung tâm

In [32]:
def read_global_products_info_v10():
    """
    Kéo bảng danh mục PRODUCTS_INFO từ BRANCH1 làm dữ liệu tham chiếu toàn cục.
    """
    config_b1 = next((c for c in BRANCH_CONFIGS if c["branch_id"] == "BRANCH1"), BRANCH_CONFIGS[0])
    schema = config_b1["schema"]
    
    query = f"""
        SELECT product_id, product_name, category, brand 
        FROM {schema}.products_info
    """
    conn = None
    try:
        conn = psycopg2.connect(
            host=config_b1["host"],
            port=config_b1["port"],
            database=config_b1["database"],
            user=config_b1["user"],
            password=config_b1["password"],
            sslmode="disable",
            connect_timeout=5
        )
        return pd.read_sql_query(query, conn)
    except Exception as e:
        print(f"❌ Câu 10 - Không thể lấy bảng danh mục PRODUCTS_INFO từ BRANCH1: {e}")
        return pd.DataFrame()
    finally:
        if conn is not None:
            conn.close()

Bộ xử lý xếp hạng Top 5 sản phẩm theo từng chi nhánh - Global Query Engine

In [33]:
def execute_distributed_query_10(top_n=5):
    local_dfs = []

    # 1. Gom dữ liệu doanh thu sản phẩm thô từ các chi nhánh phân tán
    for config in BRANCH_CONFIGS:
        df_branch = read_local_branch_product_revenue(config)
        if not df_branch.empty:
            local_dfs.append(df_branch)

    if not local_dfs:
        print("LỖI TOÀN CỤC: Không lấy được dữ liệu từ bất kỳ máy chi nhánh nào!")
        return pd.DataFrame()

    # 2. Tương đương phép UNION ALL (Bảng tạm product_revenue trong SQL)
    product_revenue_all = pd.concat(local_dfs, ignore_index=True)

    # 3. Đọc bảng danh mục sản phẩm nền từ BRANCH1
    df_products_info = read_global_products_info_v10()
    
    # 4. Sử dụng LEFT JOIN để bảo toàn dữ liệu nếu bảng danh mục của BRANCH1 bị thiếu mã sản phẩm của nhánh khác
    if not df_products_info.empty:
        merged_df = pd.merge(product_revenue_all, df_products_info, on="product_id", how="left")
        # Điền thông tin mặc định nếu sản phẩm chưa được cập nhật danh mục ở BRANCH1
        merged_df["product_name"] = merged_df["product_name"].fillna("Unknown Product")
        merged_df["category"] = merged_df["category"].fillna("Unknown Category")
        merged_df["brand"] = merged_df["brand"].fillna("Unknown Brand")
    else:
        merged_df = product_revenue_all.copy()
        merged_df["product_name"] = "Unknown Product"
        merged_df["category"] = "Unknown Category"
        merged_df["brand"] = "Unknown Brand"

    # 5. Sắp xếp dữ liệu theo từng chi nhánh và doanh thu giảm dần
    merged_df = merged_df.sort_values(by=["source_branch", "total_revenue"], ascending=[True, False])

    # 6. Tương đương RANK() OVER (PARTITION BY source_branch ORDER BY total_revenue DESC)
    # SỬA ĐỔI CHÍNH: Ép kiểu sang 'Int64' để hỗ trợ an toàn cho giá trị NaN/Null trong hệ thống phân tán
    merged_df["revenue_rank"] = (
        merged_df
        .groupby("source_branch")["total_revenue"]
        .rank(ascending=False, method="min")
        .astype("Int64") 
    )

    # 7. Tương đương mệnh đề WHERE revenue_rank <= 5
    final_top_products = merged_df[merged_df["revenue_rank"] <= top_n].copy()

    # Làm tròn giá trị tổng doanh thu 2 chữ số thập phân
    final_top_products["total_revenue"] = final_top_products["total_revenue"].round(2)

    # Tổ chức lại thứ tự hiển thị các cột chuẩn theo yêu cầu đề bài
    ordered_columns = [
        "source_branch", "product_id", "product_name", 
        "category", "brand", "total_quantity", 
        "total_revenue", "revenue_rank"
    ]
    
    return final_top_products[ordered_columns]

Khởi chạy câu hỏi truy vấn số 10 và xuất báo cáo

In [34]:
# Thực thi câu truy vấn tìm Top 5 sản phẩm doanh thu cao nhất mỗi chi nhánh
top_products_report_df = execute_distributed_query_10(top_n=5)

# Xuất kết quả trực quan ra màn hình
if not top_products_report_df.empty:
    print("\n--- BÁO CÁO BÁN HÀNG: TOP 5 SẢN PHẨM MANG LẠI DOANH THU CAO NHẤT TỪNG CHI NHÁNH ---")
    display(top_products_report_df)

-> Câu 10 - Thành công: Đã tải và tính toán xong doanh số sản phẩm từ BRANCH1 (26.51.227.155)
-> Câu 10 - Thành công: Đã tải và tính toán xong doanh số sản phẩm từ BRANCH2 (26.250.61.91)
-> Câu 10 - Thành công: Đã tải và tính toán xong doanh số sản phẩm từ BRANCH3 (26.101.109.119)

--- BÁO CÁO BÁN HÀNG: TOP 5 SẢN PHẨM MANG LẠI DOANH THU CAO NHẤT TỪNG CHI NHÁNH ---


,source_branch,product_id,product_name,category,brand,total_quantity,total_revenue,revenue_rank
18,BRANCH1,46,USB Hub 7-Port,Electronics,Sony,17112.0,4066814.20,1
21,BRANCH1,11,Wireless Mouse 2.4GHz,Electronics,Nike,16909.0,3985719.42,2
45,BRANCH1,111,Laptop Backpack Waterproof,Electronics,Xiaomi,16841.0,3949891.42,3
31,BRANCH1,20,USB-C Fast Charger 65W,Electronics,Huawei,16706.0,3910104.72,4
19,BRANCH1,39,Wireless Bluetooth Earbuds,Electronics,Xiaomi,16579.0,3901002.31,5
65,BRANCH2,346,USB-C Fast Charger 65W,Electronics,Nike,16872.0,4027717.61,1
87,BRANCH2,411,Phone Case Shockproof,Electronics,GoPro,17128.0,4001213.90,2
68,BRANCH2,426,Laptop Backpack Waterproof,Electronics,Baseus,16982.0,3973059.81,3
55,BRANCH2,402,Smart Watch Series 8,Electronics,GoPro,16769.0,3934268.27,4
74,BRANCH2,339,USB Hub 7-Port,Electronics,Apple,16687.0,3920206.83,5
